In [ ]:
import platform, subprocess, os
print("=== IBM Z / LinuxONE Hardware Verification ===")
print("Architecture :", platform.machine())
print("Processor    :", platform.processor())
print("OS Release   :", platform.platform())
if platform.machine() == "s390x":
    print("\n[CONFIRMED] Running natively on IBM LinuxONE (s390x) Mainframe!")
    print(subprocess.getoutput("lscpu | grep -E 'Architecture|Model name|CPU|Vendor|Flags'"))

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
from Backend.data_engine import build_unified_dataset
from Backend.train_benchmark import run_training_and_benchmark

df = build_unified_dataset(num_samples_per_source=10000)
metrics = run_training_and_benchmark(epochs=6, batch_size=256)

In [ ]:
import time, numpy as np, onnxruntime as ort

onnx_path = "../Backend/artifacts/aegis_moe.onnx" if os.path.exists("../Backend/artifacts/aegis_moe.onnx") else "Backend/artifacts/aegis_moe.onnx"
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
dummy_batch = np.random.randn(64, 16).astype(np.float32)

t0 = time.perf_counter()
logits, routing = sess.run(["logits", "expert_routing"], {"features": dummy_batch})
elapsed_ms = (time.perf_counter() - t0) * 1000.0

print(f"Batch (64 tx) ONNX Inference Time : {elapsed_ms:.3f} ms ({elapsed_ms/64:.4f} ms/tx)")
print("Sample Top-4 Expert Routing Weights:", np.round(routing[0], 3))